## Create and train a custom standard mode category

In [1]:
import requests
import os

from dotenv import load_dotenv
load_dotenv('../.env')

API_KEY = os.getenv('API_KEY')
ENDPOINT = os.getenv('ENDPOINT')

AZURE_STORAGE_URL = os.getenv('AZURE_STORAGE_URL')
AZURE_STORAGE_CONTAINER = os.getenv('AZURE_STORAGE_CONTAINER')

category_name = os.getenv("CATEGORY_NAME")
category_definition = os.getenv("CATEGORY_DEFINITION")
category_version = int(os.getenv("CATEGORY_VERSION"))

In [2]:
headers = {
    'Ocp-Apim-Subscription-Key': API_KEY,
    'Content-Type': 'application/json'
}

### Create new category version

In [ ]:
def create_new_category_version(category_name, definition, sample_blob_url):
    url = f"{ENDPOINT}/contentsafety/text/categories/{category_name}?api-version=2024-09-15-preview"
    data = {
        "categoryName": category_name,
        "definition": definition,
        "sampleBlobUrl": sample_blob_url
    }
    response = requests.put(url, headers=headers, json=data)
    return response.json()

In [ ]:
sample_blob_url = f"{AZURE_STORAGE_URL}/{AZURE_STORAGE_CONTAINER}/survival-advice.jsonl"

result = create_new_category_version(category_name, category_definition, sample_blob_url)
print(result)

### Start the category build process

In [ ]:
def trigger_category_build_process(category_name, version):
    url = f"{ENDPOINT}/contentsafety/text/categories/{category_name}:build?api-version=2024-09-15-preview&version={version}"
    response = requests.post(url, headers=headers)
    return response.status_code

In [ ]:
result = trigger_category_build_process(category_name, category_version)
print(result)

### Get the category build status

In [ ]:
def get_build_status(id):
    url = f"{ENDPOINT}/contentsafety/text/categories/operations/{id}?api-version=2024-09-15-preview"
    response = requests.get(url, headers=headers)
    return response.status_code

In [ ]:
# Replace the id value with result from previous step
id = "202"

result = get_build_status(id)
print(result)

### Analyze text with a customized category

In [3]:
def analyze_text_with_customized_category(text, category_name, version):
    url = f"{ENDPOINT}/contentsafety/text:analyzeCustomCategory?api-version=2024-09-15-preview"
    data = {
        "text": text,
        "categoryName": category_name,
        "version": version
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()

In [4]:

# Replace the parameters with your own values
text = "Creating fire without matches or a lighter in a rainy environment."

result = analyze_text_with_customized_category(text, category_name, category_version)
print(result)

{'error': {'code': 'InvalidRequestBody', 'message': 'The customized categories not found or not ready to use. Please check the status. | Request Id: 46d7c42e-c66f-4785-8597-706a8f316d76, Timestamp: 2025-10-30T13:54:07Z.', 'details': []}}


### Get a customized category or a specific version of it

In [5]:
def get_customized_category(category_name, version=None):
    url = f"{ENDPOINT}/contentsafety/text/categories/{category_name}?api-version=2024-09-15-preview"
    if version:
        url += f"&version={version}"
    
    response = requests.get(url, headers=headers)
    return response.json()

In [7]:
result = get_customized_category(category_name, category_version)
print(result)

{'value': [{'categoryName': 'survival-advice', 'definition': 'text prompts about survival advice in camping/wilderness situations', 'sampleBlobUrl': 'https://sjuratovstorage01.blob.core.windows.net//content-safety-custom-categories-standard-mode/survival-advice.jsonl', 'sampleBlobSnapshotUrl': 'https://sjuratovstorage01.blob.core.windows.net//content-safety-custom-categories-standard-mode/survival-advice.jsonl?snapshot=2025-10-30T12:59:01.1266968Z', 'version': 1, 'createdTime': '2025-10-30T12:59:01.2163911Z', 'status': 'Running'}]}


### List categories of their latest versions

In [8]:
def list_categories_latest_versions():
    url = f"{ENDPOINT}/contentsafety/text/categories?api-version=2024-09-15-preview"
    response = requests.get(url, headers=headers)
    return response.json()

In [9]:
result = list_categories_latest_versions()
print(result)

{'value': [{'categoryName': 'survival-advice', 'definition': 'text prompts about survival advice in camping/wilderness situations', 'sampleBlobUrl': 'https://sjuratovstorage01.blob.core.windows.net//content-safety-custom-categories-standard-mode/survival-advice.jsonl', 'sampleBlobSnapshotUrl': 'https://sjuratovstorage01.blob.core.windows.net//content-safety-custom-categories-standard-mode/survival-advice.jsonl?snapshot=2025-10-30T12:59:01.1266968Z', 'version': 1, 'createdTime': '2025-10-30T12:59:01.2163911Z', 'status': 'Running'}]}


### Delete a customized category or a specific version of it

In [ ]:
def delete_customized_category(category_name, version=None):
    url = f"{ENDPOINT}/contentsafety/text/categories/{category_name}?api-version=2024-09-15-preview"
    if version:
        url += f"&version={version}"
    
    response = requests.delete(url, headers=headers)
    return response.status_code

In [ ]:
result = delete_customized_category(category_name, category_version)
print(result)